In [7]:
import json
from pathlib import Path

In [8]:
with open('./data/target_api_resp.json') as f:
    api_resp = json.load(f)

In [9]:
# from resp
targets = []
for resp in api_resp:
    targets = [*targets, *resp.get('targets', [])]

In [10]:
# from db
#targets = []
with open('./data/kpf_cc_targets_2025A.txt') as f:
    targets = [*targets , *json.load(f)]

In [11]:
obs = []
for cc_target in targets:
    target = {
    	'target_name': (cc_target.get('target_name', 'undefined')),
    	'gaia_id': cc_target.get('gaia_id'),
    	'systemic_velocity': cc_target.get('systemic_velocity'),
    	'g_mag': cc_target.get('g_mag'),
    	'j_mag': cc_target.get('j_mag'),
    	't_eff': cc_target.get('t_eff'),
    	'ra': cc_target.get('ra'),
    	'dec': cc_target.get('dec'),
    	'pm_ra': cc_target.get('pm_ra'),
    	'pm_dec': cc_target.get('pm_dec'),
    	'epoch': cc_target.get('epoch')
        }
    observation = {
        'exposure_time': cc_target.get('nominal_exposure_time'),
        'num_exposures': cc_target.get('num_exposures_per_visit'),
        }
    
    schedule = {
    	'scheduling_mode': 'Cadence',
    	'num_visits_per_night': cc_target.get('num_visits_per_night'),
    	'num_nights_per_semester': cc_target.get('num_unique_nights_per_semester'),
    	'num_internight_cadence': cc_target.get('num_internight_cadence'),
    	'num_intranight_cadence': cc_target.get('num_intranight_cadence'),
        }
    
    metadata = {
    	'obsid': cc_target.get('obsid'),
    	'observer_name': cc_target.get('submitter'),
    	'semester': cc_target.get('semid').split('_')[0],
    	'progid': cc_target.get('semid').split('_')[1],
    	'semid': cc_target.get('semid'),
    	'history': [],
    	'tags': ['imported from legacy targets'],
        }
    
    ob = {
        'del_flag': 0,
        'metadata': metadata,
    	'target': target,
    	'observation': observation,
    	'schedule': schedule,
    	'calibration': {}
        }
    obs.append(ob)


In [12]:
with open('legacy_obs_2025_03_10.json', 'w') as f:
    json.dump(obs, f, indent=4)

# transfer instructions

1. sudo mongo observing_dev --quiet --eval 'db.kpf_cc_targets.find({semid: {$regex: '2025A_'}, del_flag: 0})' > /kpf_cc_2025A.json
2. scp file over to where this script is run
3. run script (you may need to sub out ' with " and put quotes around the keywords
4. scp legacy_obs.json dsibld@10.136.1.80:/home/dsibld/legacy_obs_2425.json  
5. head over to vm-odb2 as dsibld
6. Optional: run db.kpf_cc_observing_block_dev.deleteMany({"metadata.tags": {$in: ['imported from legacy targets']}, "metadata.semester": {$in: ["2024A", "2024B", "2025A"]}})
7. mongoimport --db observing_dev --collection kpf_observing_block_dev --jsonArray --file legacy_obs.json

In [6]:
componentNames = ['metadata', 'target', 'observation', 'schedule', 'calibration']
schemaFiles = ['metadata_schema.json', 'ob_target_schema.json', 'observation_schema.json', 'schedule_data_schema.json', 'calibration_schema.json']
def get_schema_dict():
    schemas = {}
    schemaPath = Path('./schemas/')
    for component, file in zip(componentNames, schemaFiles):
        with open(schemaPath / file) as f:
            schema = json.load(f)
        schemas[component] = schema
    return schemas

inverse_map = lambda myMap: {v: k for k, v in myMap.items()}

def make_schema_mapping():
    obSchema = get_schema_dict()
    obKeyMapping = {}
    for ckey, schema in obSchema.items():
        properties = schema['properties']
        componentMap = {}
        for key, props in properties.items():
            componentMap[props.get('translator_mapping', key)] = key
        obKeyMapping[ckey] = componentMap
    return obKeyMapping

def swap_translator_ob_to_ob_keys(ob, keyMap):
    # converts ob to use KPF Translator keys
    OB = {}
    if '_id' in ob.keys():
        OB['_id'] = ob['_id']
    OB['del_flag'] = ob['del_flag']
    components = [[key, comp] for key, comp in ob.items() if key in componentNames]
    for ckey, component in components:
        Component = {}
        for key, value in component.items():
            Component[keyMap[ckey][key]] = value
        OB[ckey] = Component
    return OB

In [7]:
OBobKeyMapping = make_schema_mapping()
obOBKeyMapping = {key: inverse_map(mp) for key, mp in OBobKeyMapping.items()}

In [8]:
OBS = []
for ob in obs:
    OB = swap_translator_ob_to_ob_keys(ob, obOBKeyMapping)
    OBS.append(OB)